<a href="https://colab.research.google.com/github/ale-nunes/LLM/blob/main/LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Visão Geral do Fluxo**
* 1.  Configuração do Ambiente e Chave de API (Google AI Studio).

* 2. Ingestão e Tratamento de Dados (Carregamento e normalização de documentos/textos).

* 3. Chunking e Indexação Vetoral (Geração de embeddings open-source com HuggingFace e armazenamento local com ChromaDB).

* 4. Recuperação (Retriever) e Teste do RAG (Busca semântica e integração com o Gemini).

Montando do Drive Para Conectar na base de dados

In [2]:
from google.colab import drive
drive.mount('/gdrive')
%cd /gdrive

Mounted at /gdrive
/gdrive


Instalação das bibliotecas de manipulação de texto, vetores e IA do Google

In [3]:
!pip install -qU langchain langchain-community sentence-transformers chromadb google-generativeai pypdf langchain-text-splitters

print("Ambiente configurado com sucesso!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.0/572.0 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2

In [ ]:
# !pip install -qU langchain-core

In [4]:
from langchain_community.document_loaders import PyPDFLoader
import os


# 2. Defina o caminho do seu PDF no Drive (substitua pelo nome exato do arquivo)

# caminho_pdf = '/gdrive/MyDrive/base vendas/Storytelling com Dados.pdf'
caminho_pdf = '/gdrive/MyDrive/base vendas/Estatistica_Aplicada_6.pdf'

# Carregando o PDF
loader = PyPDFLoader(caminho_pdf)
pages = loader.load()

print(f"PDF carregado com sucesso! Total de páginas: {len(pages)}")

/tmp/ipykernel_1318/1492626321.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


PDF carregado com sucesso! Total de páginas: 674


In [5]:
import re

def normalizar_texto(texto_bruto):
    """
    Remove quebras de linha artificiais, espaços excessivos e caracteres corrompidos.
    """
    # Substituir quebras de linha por espaço
    texto = re.sub(r'\s+', ' ', texto_bruto)
    # Remover caracteres especiais indesejados mantendo a pontuação padrão
    texto = re.sub(r'[^\w\s.,;:?!\-–]', '', texto)
    # Normalizar espaços múltiplos
    texto = re.sub(r'\s{2,}', ' ', texto).strip()
    return texto

# Aplicando a normalização em todas as páginas carregadas do PDF
texto_completo_normalizado = " ".join([normalizar_texto(page.page_content) for page in pages])

print("Exemplo do texto normalizado (primeiros 400 caracteres):")
print(texto_completo_normalizado[:400])

Exemplo do texto normalizado (primeiros 400 caracteres):
ISBN 978-85-430-0477-8 loja.pearson.com.br O objetivo de Estatística aplicada é ensinar os estudantes a utilizar o conhecimento estatístico para retratar e descrever o mundo e, a partir disso, tomar decisões fundamentadas. Totalmente revista e atualizada, esta edição mantém sua simplicidade e clareza ao apresentar os principais conceitos da estatística aplicados em situações reais por meio de estu


Importando as Bibliotecas de IA

* langchain e langchain-community: Orquestram o fluxo de RAG (divisão de textos, conexão com vetores e LLMs).

* chromadb: Banco de dados vetorial leve que roda direto na memória ou disco do Colab (open-source).

* sentence-transformers: Gera os embeddings (vetorização dos textos) rodando localmente de graça via HuggingFace.


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document


# Tratamento e Normalização dos Dados
# 1. Criando o documento formatado com o texto limpo
doc_obj = [Document(page_content=texto_completo_normalizado)]

# 2. Fragmentação inteligente (Chunking)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,   # Tamanho de cada bloco
    chunk_overlap=50  # Sobreposição para manter o contexto entre blocos
)
chunks = text_splitter.split_documents(doc_obj)
print(f"Total de fragmentos (chunks) gerados do PDF: {len(chunks)}")

# 3. Gerando Embeddings Open Source (Roda localmente na CPU/GPU do Colab)
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 4. Criando a base vetorial local com ChromaDB
vectorstore = Chroma.from_documents(chunks, embeddings_model, persist_directory="/gdrive/MyDrive/base vendas//meu_banco_chroma")
retriever = vectorstore.as_retriever(search_kwargs={"k": 5}) # Retorna os 5 trechos mais relevantes
print("Banco vetorial criado e indexado com sucesso!")

Total de fragmentos (chunks) gerados do PDF: 2070


/tmp/ipykernel_1318/1123149299.py:20: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Banco vetorial criado e indexado com sucesso!


Obtendo a Chave da API do Google AI Studio (Grátis)

* Acesse o Google AI Studio.

* Faça login com sua conta Google.

* Clique em "Get API key" (Criar chave de API) e copie a chave gerada. Ela possui um tier gratuito generoso ideal para testes e desenvolvimento.

In [7]:
from google.colab import userdata
import google.generativeai as genai

# Configurando a API do Gemini
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# Instanciando o modelo gratuito do Gemini
model = genai.GenerativeModel('gemini-3.6-flash')
print("Modelo Gemini configurado com sucesso!")

Modelo Gemini configurado com sucesso!


Executando o RAG e Testando Perguntas sobre o PDF
Agora unimos a busca semântica local baseada no seu PDF com a inteligência do Gemini.

In [8]:
def executar_rag_pdf(pergunta):
    # 1. Utilizando .invoke() para buscar os trechos mais relevantes do PDF no banco vetorial
    docs_recuperados = retriever.invoke(pergunta)
    contexto = "\n\n".join([doc.page_content for doc in docs_recuperados])

    # 2. Montar o prompt restrito ao contexto do PDF
    prompt_final = f"""
    Com base exclusivamente no conteúdo do documento PDF fornecido abaixo, responda à pergunta de forma clara, precisa e objetiva.
    Se a informação não estiver no texto, informe que não encontrou dados suficientes no documento.

    Contexto do PDF:
    {contexto}

    Pergunta: {pergunta}
    """

    # 3. Gerar a resposta utilizando o Gemini
    resposta = model.generate_content(prompt_final)
    return resposta.text

In [9]:
# --- TESTE 1 ---
pergunta_1 = "me de um resumo do que é o teste de hipotese"
print(f"Pergunta: {pergunta_1}\n")
print(f"Resposta:\n{executar_rag_pdf(pergunta_1)}\n" + "-"*50)

Pergunta: me de um resumo do que é o teste de hipotese

Resposta:
Com base exclusivamente no texto fornecido, segue o resumo do que é um **teste de hipótese**:

* **Definição:** É um processo da estatística inferencial que utiliza estatísticas amostrais para testar uma afirmação sobre o valor de um parâmetro populacional.
* **Finalidade e Uso:** É utilizado por pesquisadores (em áreas como medicina, psicologia e negócios) para a tomada de decisões sobre novos medicamentos, tratamentos, estratégias de mercado, entre outros. 
* **Funcionamento:** Como geralmente não é possível testar toda a população, retira-se uma amostra aleatória para medir os dados. A partir da distribuição amostral (supondo a hipótese nula $H_0$ como verdadeira), avalia-se se a estatística amostral é incomum ou se a média difere o suficiente para tomar a decisão de **rejeitar** ou **não rejeitar** $H_0$. Se a hipótese nula não for verdadeira, aceita-se a hipótese alternativa ($H_a$).
* **Possibilidade de Erros:** Co

In [ ]:
# # --- TESTE 2 ---
# pergunta_2 = "o que é probabilidade"
# print(f"Pergunta: {pergunta_2}\n")
# print(f"Resposta:\n{executar_rag_pdf(pergunta_2)}")